# Week 5: windows, block masks and kernels

Three experiments: what the masks look like, whether sorting makes windows work, and what each attention
kernel costs on *your* GPU.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

In [ ]:
from torch.nn.attention.flex_attention import create_block_mask

from minihep.flex.masks import mask_mod_to_dense, sliding_window_mask, sliding_window_mask_wrapped, strided_window_mask

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(mask_mod_to_dense(sliding_window_mask(16), 64, 64), cmap="Greys")
axes[0].set_title("sliding window (encoder)")
axes[1].imshow(mask_mod_to_dense(sliding_window_mask_wrapped(16, torch.tensor([64])), 64, 64), cmap="Greys")
axes[1].set_title("wrapped (phi is periodic)")
axes[2].imshow(mask_mod_to_dense(strided_window_mask(16, 16, 64), 16, 64), cmap="Greys", aspect="auto")
axes[2].set_title("strided local cross-attention (decoder)")
for ax in axes:
    ax.set_xlabel("key")
    ax.set_ylabel("query")
plt.show()
print(create_block_mask(sliding_window_mask(512), B=None, H=None, Q_LEN=4096, KV_LEN=4096, device="cpu"))

## Does sorting make windows useful?

Pair recall: the fraction of same-particle hit pairs that end up inside each other's window.

In [ ]:
from exercises.week05.ex03_locality import cell_sort_value, pair_recall, positions_from_sort_value
from minihep.data.dataset import ToyTrackingDataset

data_dir = ensure_toy_data(num_train=50 if FAST else 2000)
ds = ToyTrackingDataset(str(data_dir / "train"), {"hit": ["x"]}, {})
rng = np.random.default_rng(0)
windows = [4, 8, 16, 32, 64, 128]
curves = {"random": [], "phi": [], "eta-phi cells": []}
for w in windows:
    rec = {k: [] for k in curves}
    for i in range(10):
        hits, _ = ds.load_event(i)
        pid, n = hits.particle_id.to_numpy(), len(hits)
        rec["random"].append(pair_recall(pid, rng.permutation(n), w, n))
        rec["phi"].append(pair_recall(pid, positions_from_sort_value(hits.phi.to_numpy()), w, n))
        rec["eta-phi cells"].append(pair_recall(pid, positions_from_sort_value(cell_sort_value(hits.eta.to_numpy(), hits.phi.to_numpy(), 6)), w, n))
    for k in curves:
        curves[k].append(np.mean(rec[k]))
for k, v in curves.items():
    plt.plot(windows, v, "o-", label=k)
plt.xscale("log", base=2)
plt.xlabel("window size")
plt.ylabel("same-particle pair recall")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Kernel benchmark

Time and peak memory of one attention call versus sequence length. On CPU only the small sizes run.
On your GPU, compare full SDPA, SDPA with a dense window mask, compiled flex with a window BlockMask and
FlashAttention-2 with a window (if you installed the `fa2` environment: `pixi run -e fa2 lab`).

In [ ]:
import time

import pandas as pd
import torch.nn.functional as F
from torch.nn.attention.flex_attention import create_mask, flex_attention


def time_fn(fn, *args, warmup=3, iters=10, device="cpu"):
    # median milliseconds per call, synchronising the GPU around each call (week 8 explains why)
    sync = torch.cuda.synchronize if device == "cuda" else (lambda: None)
    for _ in range(warmup):
        fn(*args)
    times = []
    for _ in range(iters):
        sync()
        start = time.perf_counter()
        fn(*args)
        sync()
        times.append((time.perf_counter() - start) * 1000)
    return float(np.median(times))


try:
    from flash_attn import flash_attn_func
except ImportError:
    flash_attn_func = None

dtype = torch.float16 if DEVICE == "cuda" else torch.float32
flex = torch.compile(flex_attention, dynamic=False) if DEVICE == "cuda" else flex_attention
window, heads, dim = 256, 8, 32
sizes = [512, 1024, 2048] if (FAST or DEVICE == "cpu") else [1024, 2048, 4096, 8192, 16384, 32768]
rows = []
for n in sizes:
    q = torch.randn(1, heads, n, dim, device=DEVICE, dtype=dtype)
    mod = sliding_window_mask(window)
    bm = create_block_mask(mod, B=None, H=None, Q_LEN=n, KV_LEN=n, device=DEVICE)
    row = {"N": n, "sdpa full": time_fn(F.scaled_dot_product_attention, q, q, q, device=DEVICE)}
    if n <= 16384:
        dense = create_mask(mod, 1, 1, n, n, device=DEVICE)
        row["sdpa dense mask"] = time_fn(lambda: F.scaled_dot_product_attention(q, q, q, attn_mask=dense), device=DEVICE)
    row["flex window"] = time_fn(lambda: flex(q, q, q, block_mask=bm), device=DEVICE)
    if flash_attn_func is not None and DEVICE == "cuda":
        qf = q.transpose(1, 2).contiguous()
        row["flash window"] = time_fn(lambda: flash_attn_func(qf, qf, qf, window_size=(window // 2, window // 2)), device=DEVICE)
    rows.append(row)
table = pd.DataFrame(rows).set_index("N")
print(table.round(3))
table.plot(marker="o", logx=True, logy=True, ylabel="ms per call")
plt.show()

## Your turn

1. In the table, where does windowed attention start to beat full attention? A dense mask still makes SDPA
   visit every (query, key) pair, so what speed-up should you expect from it, and does your table agree?
   (Run on an otherwise idle GPU, and close other notebooks first.)
2. Change `window` to 64 and 1024. How does flex's time scale with the window at fixed N?
3. Read `per_head_window_mask_mod` in hepattn's `flex/per_head_window.py`. What does it capture in its
   closure that `sliding_window_mask` does not, and why must it be rebuilt for every event?